# 🎲 Module 4.3: Monte Carlo Methods for Image Processing

## Learning from Complete Episodes of Image Transformations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_04_Basic_RL_Algorithms/03_Monte_Carlo_Methods/monte_carlo_methods.ipynb)

---

## 🎯 Learning Objectives
1. Understand model-free learning via Monte Carlo (MC) sampling
2. Derive and implement **First-Visit MC** and **Every-Visit MC** prediction
3. Implement **MC control** with $\epsilon$-greedy policies (GLIE)
4. Derive **importance sampling** for off-policy MC evaluation
5. Apply MC methods to learn optimal image processing pipelines

**Prerequisites:** MDP formulation, Dynamic Programming (Module 4.2)

---

In [ ]:
# Setup - Install dependencies (Google Colab)
!pip install numpy matplotlib opencv-python-headless pillow scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print("✅ All libraries loaded successfully!")

In [ ]:
# Download REAL open-source dataset
import torchvision
import numpy as np
from PIL import Image

# CIFAR-10 for image filter selection environment
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
# Take small subset for fast RL experiments
real_images = [np.array(cifar10[i][0]) for i in range(200)]
print(f"✅ CIFAR-10 loaded: Using {len(real_images)} real photos for RL environment")
print(f"   Image shape: {real_images[0].shape}")
print(f"   These will be our 'states' that the RL agent processes!")

## Deep Derivation: Monte Carlo Methods — From Sampling to Convergence

### Step 1: Monte Carlo Estimation (Foundation)
By the **Strong Law of Large Numbers**, the sample mean converges almost surely:
$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i \xrightarrow{a.s.} E[X] \quad \text{as } n \to \infty$$

**Proof sketch:** For i.i.d. random variables with $E[X^4] < \infty$:
$$E[(\bar{X}_n - \mu)^4] = O(1/n^2)$$
By Borel-Cantelli: $\sum_n P(|\bar{X}_n - \mu| > \epsilon) < \infty$, so $\bar{X}_n \to \mu$ a.s. $\blacksquare$

### Step 2: First-Visit MC Prediction — Unbiased Estimator

**Algorithm:** For each episode, record the return $G_t$ for the FIRST visit to state $s$.
$$V(s) \leftarrow \text{average}(\{G_t : S_t = s, \text{first visit in episode}\})$$

**Proof of unbiasedness:**
Each $G_t$ is an i.i.d. sample of the return starting from $s$ under policy $\pi$:
$$E[G_t | S_t = s] = V^\pi(s)$$

By SLLN: $\frac{1}{N}\sum_{i=1}^N G_t^{(i)} \to V^\pi(s)$ as $N \to \infty$. $\blacksquare$

### Step 3: Every-Visit MC — Also Convergent (But Biased for Finite Samples)

**Theorem:** Every-visit MC is biased for finite $N$ but consistent (converges as $N \to \infty$).

**Proof of bias:** Within a single episode, multiple visits to state $s$ produce correlated returns (they share future trajectory segments). For finite samples, this correlation biases the estimate. However, as the number of episodes $\to \infty$, the bias vanishes because each episode is independent. $\blacksquare$

### Step 4: Incremental Update Rule (Memory-Efficient MC)
Instead of storing all returns, update incrementally:
$$V(s) \leftarrow V(s) + \frac{1}{N(s)}\left[G_t - V(s)\right]$$

**Derivation:**
Let $V_n = \frac{1}{n}\sum_{i=1}^n G_i$. Then:
$$V_{n+1} = \frac{1}{n+1}\sum_{i=1}^{n+1} G_i = \frac{1}{n+1}\left(nV_n + G_{n+1}\right) = V_n + \frac{1}{n+1}(G_{n+1} - V_n) \quad \blacksquare$$

With constant step size $\alpha$ instead of $1/N(s)$:
$$V(s) \leftarrow V(s) + \alpha[G_t - V(s)]$$

This gives an exponentially weighted moving average with effective window $\sim 1/\alpha$.

### Step 5: MC Control with $\epsilon$-Greedy (GLIE Convergence)

**GLIE (Greedy in the Limit with Infinite Exploration):**
1. All state-action pairs visited infinitely often: $\lim_{k\to\infty} N_k(s,a) = \infty$
2. Policy converges to greedy: $\lim_{k\to\infty} \pi_k(a|s) = \mathbf{1}[a = \arg\max_{a'} Q(s,a')]$

**Theorem:** GLIE MC control converges to $Q^*$.

**Proof:**
- By SLLN: $Q_k(s,a) \to Q^{\pi_k}(s,a)$ as visits $\to \infty$ (MC evaluation converges)
- By Policy Improvement Theorem: the greedy policy w.r.t. $Q_k$ improves the policy
- $\epsilon_k \to 0$ ensures eventual greedy behavior → optimality $\blacksquare$

### Step 6: Importance Sampling for Off-Policy MC

**Problem:** Evaluate $V^\pi$ using episodes generated by behavior policy $b \neq \pi$.

**Importance sampling ratio for an episode:**
$$\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$$

**Proof of unbiasedness (ordinary IS):**
$$E_b[\rho_{t:T-1} G_t] = E_\pi[G_t] = V^\pi(S_t)$$

**Derivation:**
$$E_b[\rho G] = \sum_{\tau} P_b(\tau) \cdot \frac{P_\pi(\tau)}{P_b(\tau)} \cdot G(\tau) = \sum_\tau P_\pi(\tau) \cdot G(\tau) = E_\pi[G] \quad \blacksquare$$

**Weighted importance sampling (lower variance):**
$$V(s) = \frac{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1} G_t}{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1}}$$

This is biased but consistent, and typically has much lower variance than ordinary IS.

### Step 7: Variance Analysis
**Ordinary IS variance:** Can be infinite if $\pi/b$ has heavy tails!

$$\text{Var}_b[\rho G] = E_b[\rho^2 G^2] - (E_b[\rho G])^2$$

The $\rho^2$ term can explode when $b(a|s) \ll \pi(a|s)$ for some $(s,a)$.

### RL Connection: MC for Image Processing
MC methods learn WITHOUT a model of the environment:
- Run complete episodes of image enhancement (apply sequence of filters until quality converges)
- Record the total quality improvement (return)
- Average returns to estimate value of each image state
- No need to know the exact effect of filters in advance!

## 1. Why Monte Carlo?

### 1.1 DP Limitations

Dynamic Programming requires:
- Complete knowledge of $p(s', r | s, a)$ — the environment model
- Sweeps over all states — expensive for large state spaces

### 1.2 Monte Carlo Advantage

MC methods learn from **experience** (sampled episodes):
- **Model-free**: No need for $p(s', r | s, a)$
- **Episodic**: Only requires episodes to eventually terminate
- **No bootstrapping**: Uses actual returns, not estimates of future values
- **Unbiased** (first-visit): Estimates converge to true values

### 1.3 Key Idea

Average the **actual returns** observed after visiting a state:

$$v_\pi(s) \approx \frac{1}{N(s)} \sum_{i=1}^{N(s)} G_t^{(i)}$$

where $G_t^{(i)}$ is the return from the $i$-th visit to state $s$.

## 2. Monte Carlo Prediction

### 2.1 The Return

For an episode $S_0, A_0, R_1, S_1, A_1, R_2, \ldots, S_{T-1}, A_{T-1}, R_T$:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots + \gamma^{T-t-1} R_T = \sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1}$$

### 2.2 First-Visit MC Prediction

For each state $s$, average the returns from the **first** visit to $s$ in each episode:

$$V(s) = \frac{\sum_{i=1}^{N_{\text{episodes}}} G_t^{(i)} \cdot \mathbb{1}[S_t^{(i)} = s, \text{first visit}]}{\sum_{i=1}^{N_{\text{episodes}}} \mathbb{1}[S_t^{(i)} = s, \text{first visit}]}$$

**Properties:**
- **Unbiased**: $\mathbb{E}[V(s)] = v_\pi(s)$
- **Consistent**: $V(s) \xrightarrow{a.s.} v_\pi(s)$ as $N(s) \to \infty$ (Law of Large Numbers)
- **Variance**: $\text{Var}[V(s)] = \text{Var}[G_t | S_t = s] / N(s)$

### 2.3 Every-Visit MC Prediction

Average returns from **every** visit to $s$:

$$V(s) = \frac{\sum_{i=1}^{N_{\text{episodes}}} \sum_{t=0}^{T_i - 1} G_t^{(i)} \cdot \mathbb{1}[S_t^{(i)} = s]}{\sum_{i=1}^{N_{\text{episodes}}} \sum_{t=0}^{T_i - 1} \mathbb{1}[S_t^{(i)} = s]}$$

**Properties:**
- **Biased** (within episodes, visits are correlated) but **consistent**
- Often has **lower variance** than first-visit MC in practice

### 2.4 Incremental Mean Update

Instead of storing all returns, update incrementally:

$$V(S_t) \leftarrow V(S_t) + \alpha \left[G_t - V(S_t)\right]$$

With $\alpha = 1/N(S_t)$ this gives the exact sample mean. A constant $\alpha$ tracks non-stationary environments.

In [ ]:
class ImageProcessingEnv:
    """Episodic environment for sequential image processing.
    
    State: (contrast_level, sharpness_level, noise_level) each in {0,...,n_levels-1}
    Actions: Image filters
    Episode ends after max_steps or 'Done' action.
    """
    
    def __init__(self, n_levels=5, max_steps=8):
        self.n_levels = n_levels
        self.max_steps = max_steps
        self.action_names = ['Sharpen', 'Denoise', 'Contrast+', 'Hist Eq', 'Edge Enh', 'Done']
        self.n_actions = len(self.action_names)
        self.state = None
        self.steps = 0
    
    def reset(self, start_state=None):
        if start_state is not None:
            self.state = list(start_state)
        else:
            self.state = [
                np.random.randint(0, 3),  # Low contrast
                np.random.randint(0, 2),  # Low sharpness
                np.random.randint(2, self.n_levels)  # High noise
            ]
        self.steps = 0
        return tuple(self.state)
    
    def step(self, action):
        self.steps += 1
        c, s, n = self.state
        
        # Done action or max steps
        if action == 5 or self.steps >= self.max_steps:
            quality = (c + s + (self.n_levels - 1 - n)) / (3 * (self.n_levels - 1))
            return tuple(self.state), quality * 5, True
        
        reward = -0.05  # Step cost
        
        if action == 0:  # Sharpen
            self.state[1] = min(s + 1, self.n_levels - 1)
            if np.random.rand() < 0.3:
                self.state[2] = min(n + 1, self.n_levels - 1)
            reward += 0.3 if s < self.n_levels - 1 else -0.1
        elif action == 1:  # Denoise
            self.state[2] = max(n - 1, 0)
            if np.random.rand() < 0.2:
                self.state[1] = max(s - 1, 0)
            reward += 0.3 if n > 0 else -0.1
        elif action == 2:  # Contrast+
            self.state[0] = min(c + 1, self.n_levels - 1)
            reward += 0.3 if c < self.n_levels - 1 else -0.2
        elif action == 3:  # Histogram Eq
            self.state[0] = min(c + 1, self.n_levels - 1)
            if np.random.rand() < 0.5:
                self.state[2] = max(n - 1, 0)
            reward += 0.2
        elif action == 4:  # Edge Enhance
            self.state[1] = min(s + 2, self.n_levels - 1)
            self.state[2] = min(n + 1, self.n_levels - 1)
            reward += 0.1
        
        noise_reward = np.random.normal(0, 0.05)
        return tuple(self.state), reward + noise_reward, False
    
    def state_to_key(self, state):
        return tuple(state)

env = ImageProcessingEnv(n_levels=5, max_steps=8)
print(f"Environment: {env.n_levels}^3 = {env.n_levels**3} non-terminal states")
print(f"Actions: {env.action_names}")
print(f"Max episode length: {env.max_steps}")

In [ ]:
def generate_episode(env, policy_fn):
    """Generate a full episode using a policy function.
    
    policy_fn(state) -> action
    Returns list of (state, action, reward) tuples.
    """
    episode = []
    state = env.reset()
    done = False
    
    while not done:
        action = policy_fn(state)
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state
    
    return episode

def random_policy(state):
    return np.random.randint(env.n_actions)

# Generate a sample episode
sample_episode = generate_episode(env, random_policy)
print("Sample episode (random policy):")
print(f"{'Step':>4} | {'State (C,S,N)':>15} | {'Action':>12} | {'Reward':>8}")
print("-" * 50)
for t, (s, a, r) in enumerate(sample_episode):
    print(f"{t:>4} | {str(s):>15} | {env.action_names[a]:>12} | {r:>8.3f}")

# Compute return
gamma = 0.95
G = sum(gamma**t * r for t, (_, _, r) in enumerate(sample_episode))
print(f"\nTotal return G₀ (γ={gamma}): {G:.3f}")

## 3. First-Visit MC Prediction

### Algorithm

$$\boxed{\textbf{First-Visit MC Prediction}}$$

**Input:** Policy $\pi$ to evaluate

Initialize $V(s) = 0$, $N(s) = 0$ for all $s$

**Repeat** for each episode:
1. Generate episode following $\pi$: $S_0, A_0, R_1, \ldots, S_{T-1}, A_{T-1}, R_T$
2. $G \leftarrow 0$
3. **For** $t = T-1, T-2, \ldots, 0$:
   - $G \leftarrow \gamma G + R_{t+1}$
   - If $S_t$ not in $\{S_0, \ldots, S_{t-1}\}$ (first visit):
     - $N(S_t) \leftarrow N(S_t) + 1$
     - $V(S_t) \leftarrow V(S_t) + \frac{1}{N(S_t)}[G - V(S_t)]$

In [ ]:
def first_visit_mc_prediction(env, policy_fn, n_episodes=5000, gamma=0.95):
    """First-visit MC prediction for state values."""
    V = defaultdict(float)
    N = defaultdict(int)
    v_history = []  # Track estimates for convergence analysis
    
    for ep in range(n_episodes):
        episode = generate_episode(env, policy_fn)
        
        G = 0
        visited = set()
        
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            if state not in visited:
                visited.add(state)
                N[state] += 1
                V[state] += (G - V[state]) / N[state]
        
        if ep % 100 == 0:
            v_history.append(dict(V))
    
    return dict(V), dict(N), v_history

def every_visit_mc_prediction(env, policy_fn, n_episodes=5000, gamma=0.95):
    """Every-visit MC prediction for state values."""
    V = defaultdict(float)
    N = defaultdict(int)
    v_history = []
    
    for ep in range(n_episodes):
        episode = generate_episode(env, policy_fn)
        
        G = 0
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            N[state] += 1
            V[state] += (G - V[state]) / N[state]
        
        if ep % 100 == 0:
            v_history.append(dict(V))
    
    return dict(V), dict(N), v_history

print("Running First-Visit MC...")
V_fv, N_fv, hist_fv = first_visit_mc_prediction(env, random_policy, n_episodes=10000)
print(f"  States visited: {len(V_fv)}")

print("Running Every-Visit MC...")
V_ev, N_ev, hist_ev = every_visit_mc_prediction(env, random_policy, n_episodes=10000)
print(f"  States visited: {len(V_ev)}")

In [ ]:
# Compare first-visit vs every-visit for specific states
track_states = [(0, 0, 4), (2, 2, 2), (4, 4, 0), (1, 0, 3)]
state_labels = ['Worst (0,0,4)', 'Medium (2,2,2)', 'Best (4,4,0)', 'Low (1,0,3)']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (state, label) in enumerate(zip(track_states, state_labels)):
    ax = axes[idx // 2, idx % 2]
    
    fv_values = [h.get(state, 0) for h in hist_fv]
    ev_values = [h.get(state, 0) for h in hist_ev]
    
    episodes = [i * 100 for i in range(len(fv_values))]
    
    ax.plot(episodes, fv_values, 'b-', linewidth=2, label='First-Visit MC', alpha=0.8)
    ax.plot(episodes, ev_values, 'r-', linewidth=2, label='Every-Visit MC', alpha=0.8)
    ax.set_xlabel('Episode')
    ax.set_ylabel('V(s)')
    ax.set_title(f'State {label}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('First-Visit vs Every-Visit MC: Value Convergence', fontsize=14)
plt.tight_layout()
plt.show()

# Scatter plot comparison
common_states = set(V_fv.keys()) & set(V_ev.keys())
fv_vals = [V_fv[s] for s in common_states]
ev_vals = [V_ev[s] for s in common_states]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(fv_vals, ev_vals, alpha=0.5, s=20)
lims = [min(min(fv_vals), min(ev_vals)) - 0.5, max(max(fv_vals), max(ev_vals)) + 0.5]
ax.plot(lims, lims, 'r--', linewidth=2, label='y = x')
ax.set_xlabel('First-Visit MC Value')
ax.set_ylabel('Every-Visit MC Value')
ax.set_title('First-Visit vs Every-Visit MC Values\n(Should lie near diagonal)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 4. MC Control: Finding Optimal Policies

### 4.1 From Prediction to Control

To do control, we estimate **action-value** functions $Q(s, a)$ instead of $V(s)$:

$$Q(s, a) \approx \frac{1}{N(s,a)} \sum_{i=1}^{N(s,a)} G_t^{(i)} \quad \text{where } S_t = s, A_t = a$$

### 4.2 The Exploration Problem

A deterministic greedy policy may never explore some $(s, a)$ pairs. Solutions:

1. **Exploring starts**: Every $(s, a)$ pair has nonzero probability of being the episode start
2. **$\epsilon$-soft policies**: $\pi(a|s) \geq \epsilon / |\mathcal{A}|$ for all $a$

### 4.3 GLIE (Greedy in the Limit with Infinite Exploration)

A sequence of policies $\{\pi_k\}$ is GLIE if:

1. All state-action pairs are explored infinitely often: $\lim_{k \to \infty} N_k(s,a) = \infty$
2. The policy converges to greedy: $\pi_k(s) \xrightarrow{k \to \infty} \arg\max_a Q_k(s,a)$

**Example:** $\epsilon$-greedy with $\epsilon_k = 1/k$ is GLIE.

### 4.4 On-Policy MC Control Algorithm

$$\boxed{\textbf{On-Policy First-Visit MC Control (ε-greedy)}}$$

Initialize $Q(s, a)$ arbitrarily, $\epsilon \leftarrow 1.0$

**Repeat** for each episode:
1. Generate episode using $\epsilon$-greedy policy derived from $Q$
2. For each $(S_t, A_t)$ first appearing in the episode:
   - $G \leftarrow$ return following first occurrence
   - $N(S_t, A_t) \leftarrow N(S_t, A_t) + 1$
   - $Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \frac{1}{N(S_t,A_t)}[G - Q(S_t, A_t)]$
3. Decay $\epsilon$

In [ ]:
def mc_control_epsilon_greedy(env, n_episodes=20000, gamma=0.95,
                               epsilon_start=1.0, epsilon_min=0.01, epsilon_decay=0.9995):
    """On-policy first-visit MC control with epsilon-greedy."""
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    N = defaultdict(lambda: np.zeros(env.n_actions))
    epsilon = epsilon_start
    
    episode_rewards = []
    epsilon_history = []
    
    for ep in range(n_episodes):
        # Generate episode with current epsilon-greedy policy
        def eps_greedy_policy(state):
            if np.random.rand() < epsilon:
                return np.random.randint(env.n_actions)
            return np.argmax(Q[state])
        
        episode = generate_episode(env, eps_greedy_policy)
        
        # Compute returns and update Q
        G = 0
        visited = set()
        total_reward = sum(r for _, _, r in episode)
        
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            sa = (state, action)
            if sa not in visited:
                visited.add(sa)
                N[state][action] += 1
                Q[state][action] += (G - Q[state][action]) / N[state][action]
        
        episode_rewards.append(total_reward)
        epsilon_history.append(epsilon)
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    # Extract greedy policy
    policy = {}
    for state in Q:
        policy[state] = np.argmax(Q[state])
    
    return dict(Q), policy, episode_rewards, epsilon_history

print("Running MC Control...")
Q_mc, policy_mc, rewards_mc, eps_hist = mc_control_epsilon_greedy(
    env, n_episodes=30000, gamma=0.95
)
print(f"✅ Done! States in Q-table: {len(Q_mc)}")

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Smoothed reward
window = 500
smoothed = np.convolve(rewards_mc, np.ones(window)/window, mode='valid')
axes[0].plot(smoothed, 'b-', linewidth=1.5)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Episode Reward (smoothed)')
axes[0].set_title('MC Control: Learning Curve')
axes[0].grid(True, alpha=0.3)

# Epsilon decay
axes[1].plot(eps_hist, 'r-', linewidth=1.5)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('ε')
axes[1].set_title('Exploration Rate Decay')
axes[1].grid(True, alpha=0.3)

# Q-value distribution for a sample state
sample_state = (1, 1, 3)
if sample_state in Q_mc:
    q_vals = Q_mc[sample_state]
    colors = ['#e74c3c' if i != np.argmax(q_vals) else '#2ecc71' for i in range(len(q_vals))]
    axes[2].bar(range(env.n_actions), q_vals, color=colors)
    axes[2].set_xticks(range(env.n_actions))
    axes[2].set_xticklabels(env.action_names, rotation=30, ha='right')
    axes[2].set_ylabel('Q(s, a)')
    axes[2].set_title(f'Q-values for State {sample_state}\n(Green = best action)')
    axes[2].grid(True, alpha=0.3)

plt.suptitle('Monte Carlo Control Results', fontsize=14)
plt.tight_layout()
plt.show()

# Show learned policy for some states
print("\nLearned Policy (selected states):")
print(f"{'State (C,S,N)':>15} | {'Best Action':>12} | {'Q-values':>40}")
print("-" * 75)
for c in range(0, 5, 2):
    for s in range(0, 5, 2):
        for n in range(0, 5, 2):
            state = (c, s, n)
            if state in Q_mc:
                best_a = np.argmax(Q_mc[state])
                q_str = ', '.join(f'{v:.2f}' for v in Q_mc[state])
                print(f"{str(state):>15} | {env.action_names[best_a]:>12} | [{q_str}]")

## 5. Importance Sampling for Off-Policy MC

### 5.1 The Off-Policy Problem

We want to evaluate a **target policy** $\pi$ using episodes generated by a **behavior policy** $b$.

Requirement: **coverage** — $\pi(a|s) > 0 \implies b(a|s) > 0$ for all $s, a$.

### 5.2 Importance Sampling Ratio

The relative probability of a trajectory under $\pi$ vs $b$:

$$\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k | S_k)}{b(A_k | S_k)}$$

Then $\mathbb{E}_b[\rho_{t:T-1} G_t] = \mathbb{E}_\pi[G_t] = v_\pi(S_t)$.

### 5.3 Ordinary Importance Sampling

$$V_{\text{OIS}}(s) = \frac{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1} G_t}{|\mathcal{T}(s)|}$$

where $\mathcal{T}(s)$ is the set of time steps where $s$ is visited (first-visit or every-visit).

**Properties:**
- **Unbiased**: $\mathbb{E}[V_{\text{OIS}}(s)] = v_\pi(s)$
- **High variance**: $\rho$ can be very large or very small

### 5.4 Weighted Importance Sampling

$$V_{\text{WIS}}(s) = \frac{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1} G_t}{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1}}$$

**Properties:**
- **Biased** (but asymptotically unbiased)
- **Much lower variance** than ordinary IS — preferred in practice

### 5.5 Variance Comparison

Ordinary IS has variance that can be **unbounded** even with bounded rewards, because $\rho$ can grow exponentially with episode length.

Weighted IS has bounded variance: the weights normalize, bounding the effective sample size.

In [ ]:
def off_policy_mc_prediction(env, target_policy_fn, behavior_policy_fn,
                              behavior_probs_fn, target_probs_fn,
                              n_episodes=10000, gamma=0.95):
    """Off-policy MC prediction using weighted importance sampling."""
    V_ois = defaultdict(float)  # Ordinary IS
    V_wis = defaultdict(float)  # Weighted IS
    C_wis = defaultdict(float)  # Cumulative weights for WIS
    N_ois = defaultdict(int)    # Count for OIS
    
    ois_history = []
    wis_history = []
    
    for ep in range(n_episodes):
        episode = generate_episode(env, behavior_policy_fn)
        
        G = 0
        W = 1.0  # Importance weight
        
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            # Importance ratio for this step
            pi_prob = target_probs_fn(state, action)
            b_prob = behavior_probs_fn(state, action)
            
            if b_prob == 0:
                break
            
            W *= pi_prob / b_prob
            
            if W == 0:
                break
            
            # Ordinary IS update
            N_ois[state] += 1
            V_ois[state] += (W * G - V_ois[state]) / N_ois[state]
            
            # Weighted IS update
            C_wis[state] += W
            if C_wis[state] > 0:
                V_wis[state] += (W / C_wis[state]) * (G - V_wis[state])
        
        if ep % 200 == 0:
            ois_history.append(dict(V_ois))
            wis_history.append(dict(V_wis))
    
    return dict(V_ois), dict(V_wis), ois_history, wis_history

# Define target policy (greedy from MC control) and behavior policy (random)
def target_policy(state):
    if state in Q_mc:
        return np.argmax(Q_mc[state])
    return np.random.randint(env.n_actions)

def target_probs(state, action):
    if state in Q_mc:
        return 1.0 if action == np.argmax(Q_mc[state]) else 0.0
    return 1.0 / env.n_actions

def behavior_probs(state, action):
    return 1.0 / env.n_actions

print("Running Off-Policy MC with Importance Sampling...")
V_ois, V_wis, ois_hist, wis_hist = off_policy_mc_prediction(
    env, target_policy, random_policy, behavior_probs, target_probs,
    n_episodes=15000, gamma=0.95
)
print(f"✅ Done! States estimated: OIS={len(V_ois)}, WIS={len(V_wis)}")

In [ ]:
# Compare OIS vs WIS convergence
track_state = (1, 1, 3)

ois_vals = [h.get(track_state, 0) for h in ois_hist]
wis_vals = [h.get(track_state, 0) for h in wis_hist]
x_axis = [i * 200 for i in range(len(ois_vals))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x_axis, ois_vals, 'b-', linewidth=1.5, alpha=0.7, label='Ordinary IS')
axes[0].plot(x_axis, wis_vals, 'r-', linewidth=2, label='Weighted IS')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel(f'V({track_state})')
axes[0].set_title(f'Off-Policy MC: Convergence for State {track_state}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Variance comparison across states
common = set(V_ois.keys()) & set(V_wis.keys())
if len(common) > 5:
    ois_values = [V_ois[s] for s in common]
    wis_values = [V_wis[s] for s in common]
    
    axes[1].scatter(ois_values, wis_values, alpha=0.4, s=15)
    lims = [min(min(ois_values), min(wis_values)) - 1,
            max(max(ois_values), max(wis_values)) + 1]
    axes[1].plot(lims, lims, 'r--', linewidth=2)
    axes[1].set_xlabel('Ordinary IS Value')
    axes[1].set_ylabel('Weighted IS Value')
    axes[1].set_title('OIS vs WIS Estimates\n(WIS is more stable)')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Complete Example: Optimal Image Enhancement Pipeline

Let's use our learned MC policy to enhance a degraded synthetic image step by step.

In [ ]:
def simulate_pipeline(env, policy, start_state, max_steps=10):
    """Run the learned policy and record the trajectory."""
    state = env.reset(start_state)
    trajectory = [(state, None, 0)]
    total_reward = 0
    
    for step in range(max_steps):
        if state in policy:
            action = policy[state]
        else:
            action = 5  # Done
        
        next_state, reward, done = env.step(action)
        total_reward += reward
        trajectory.append((next_state, action, reward))
        
        if done:
            break
        state = next_state
    
    return trajectory, total_reward

# Run optimal policy vs random policy from worst state
start = (0, 0, 4)

# Optimal
traj_opt, reward_opt = simulate_pipeline(env, policy_mc, start)

# Random (average over many runs)
random_rewards = []
for _ in range(1000):
    _, rr = simulate_pipeline(env, {}, start)  # Empty policy → defaults to Done
    random_rewards.append(rr)

print("Optimal Pipeline from Worst State (0, 0, 4):")
print("=" * 60)
for i, (state, action, reward) in enumerate(traj_opt):
    if action is not None:
        print(f"  Step {i}: State {state} ← {env.action_names[action]} (reward: {reward:.3f})")
    else:
        print(f"  Start: State {state}")

print(f"\nOptimal policy total reward: {reward_opt:.3f}")
print(f"Random policy avg reward:    {np.mean(random_rewards):.3f} ± {np.std(random_rewards):.3f}")

# Visualize the pipeline
fig, axes = plt.subplots(1, len(traj_opt), figsize=(3 * len(traj_opt), 3))
if len(traj_opt) == 1:
    axes = [axes]

for i, (state, action, reward) in enumerate(traj_opt):
    c, s, n = state if len(state) == 3 else (0, 0, 0)
    # Create a simple quality visualization
    quality_img = np.random.rand(50, 50) * (n / 4) * 0.3  # Noise proportional to n
    quality_img += (c / 4) * 0.5  # Contrast
    quality_img = np.clip(quality_img, 0, 1)
    
    axes[i].imshow(quality_img, cmap='gray', vmin=0, vmax=1)
    title = f'C={c} S={s} N={n}'
    if action is not None:
        title = f'← {env.action_names[action]}\n{title}'
    else:
        title = f'Start\n{title}'
    axes[i].set_title(title, fontsize=9)
    axes[i].axis('off')

plt.suptitle('Optimal Image Enhancement Pipeline (MC Control)', fontsize=13)
plt.tight_layout()
plt.show()

---

## 📝 Summary

In this notebook, we covered:

1. **Monte Carlo Prediction** — estimating $v_\pi$ from sampled returns without a model
2. **First-Visit vs Every-Visit MC** — unbiased vs consistent estimators
3. **MC Control** — finding optimal policies with $\epsilon$-greedy exploration (GLIE)
4. **Importance Sampling** — off-policy evaluation using trajectory reweighting
5. **Image Processing Pipeline** — MC control applied to sequential filter selection

### 🔑 Key Equations

| Concept | Formula |
|---------|----------|
| Return | $G_t = \sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1}$ |
| MC Update | $V(S_t) \leftarrow V(S_t) + \alpha[G_t - V(S_t)]$ |
| IS Ratio | $\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$ |
| Weighted IS | $V(s) = \frac{\sum_t \rho_{t:T-1} G_t}{\sum_t \rho_{t:T-1}}$ |

### MC vs DP

| Property | DP | Monte Carlo |
|----------|----|-----------|
| Model required | Yes | No |
| Bootstrapping | Yes | No |
| Episodic only | No | Yes |
| Variance | Low | High |
| Bias | None | None (first-visit) |

---

## ➡️ Next

In the next notebook, we'll explore **TD Learning and SARSA** — methods that combine the model-free advantage of MC with the bootstrapping efficiency of DP.